<a href="https://colab.research.google.com/github/IceMeenaCreator/2025_langchain_front_todoList/blob/main/climate_assetpricing_v0_1_0523.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **0단계. Colab 환경 설정**

In [2]:
# ============================================================
# 0. Colab 환경 설정
# ============================================================

!pip install linearmodels openpyxl -q

import os
import re
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import statsmodels.api as sm

from pathlib import Path
from scipy import stats
from linearmodels.panel import PanelOLS

from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


# **1단계. 경로 설정**

In [3]:
# ============================================================
# 1. 경로 설정
# ============================================================

PROJECT_DIR = Path("/content/drive/MyDrive/박사논문/논문_climatefinance/data_climatefinance")  # 본인 구글 드라이브 폴더명으로 수정
DATA_DIR = PROJECT_DIR
RESULT_DIR = PROJECT_DIR / "results_colab"
RESULT_DIR.mkdir(parents=True, exist_ok=True)

FIRM_FILE = DATA_DIR / "data_company_nonfinancial_asset_pricing_monthly_panel_2015_2025.csv"
NEWS_FILE = DATA_DIR / "index_news.csv"

print("FIRM_FILE:", FIRM_FILE)
print("NEWS_FILE:", NEWS_FILE)
print("RESULT_DIR:", RESULT_DIR)

FIRM_FILE: /content/drive/MyDrive/박사논문/논문_climatefinance/data_climatefinance/data_company_nonfinancial_asset_pricing_monthly_panel_2015_2025.csv
NEWS_FILE: /content/drive/MyDrive/박사논문/논문_climatefinance/data_climatefinance/index_news.csv
RESULT_DIR: /content/drive/MyDrive/박사논문/논문_climatefinance/data_climatefinance/results_colab


# **2단계. 보조 함수 정의**

In [4]:
# ============================================================
# 2. 보조 함수
# ============================================================

def read_csv_safely(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"파일을 찾을 수 없습니다: {path}")

    for enc in ["utf-8-sig", "utf-8", "cp949", "euc-kr"]:
        try:
            return pd.read_csv(path, encoding=enc, low_memory=False)
        except Exception:
            pass

    raise ValueError(f"CSV 인코딩 확인 필요: {path}")


def clean_columns(df):
    df = df.copy()
    df.columns = (
        df.columns.astype(str)
        .str.strip()
        .str.replace("\n", "", regex=False)
        .str.replace("\t", "", regex=False)
        .str.replace(" ", "_", regex=False)
    )
    return df


def find_col(df, candidates, required=False, label=""):
    """
    후보 컬럼명 중 실제 존재하는 컬럼을 찾음.
    대소문자 차이와 일부 축약도 허용.
    """
    cols = list(df.columns)
    lower_map = {c.lower(): c for c in cols}

    for cand in candidates:
        if cand in cols:
            return cand
        if cand.lower() in lower_map:
            return lower_map[cand.lower()]

    # 부분 일치
    for cand in candidates:
        cand_low = cand.lower()
        for c in cols:
            if cand_low in c.lower():
                return c

    if required:
        raise KeyError(f"필수 컬럼을 찾지 못했습니다: {label}, 후보={candidates}")

    return None


def to_year_month(df):
    df = df.copy()

    if "year_month" in df.columns:
        df["year_month"] = df["year_month"].astype(str).str.slice(0, 7)
        return df

    if "date" in df.columns:
        df["year_month"] = pd.to_datetime(df["date"], errors="coerce").dt.to_period("M").astype(str)
        return df

    if "published_at" in df.columns:
        df["year_month"] = pd.to_datetime(df["published_at"], errors="coerce").dt.to_period("M").astype(str)
        return df

    if {"year", "month"}.issubset(df.columns):
        m = df["month"].astype(str)

        # month가 YYYY-MM 형태인 경우
        if m.str.contains("-").any():
            df["year_month"] = m.str.slice(0, 7)
        else:
            df["year_month"] = (
                df["year"].astype(int).astype(str)
                + "-"
                + df["month"].astype(int).astype(str).str.zfill(2)
            )
        return df

    raise KeyError("year_month를 만들 수 없습니다. year/month/date/published_at 컬럼을 확인하세요.")


def zscore(s):
    s = pd.to_numeric(s, errors="coerce")
    sd = s.std(skipna=True)
    if pd.isna(sd) or sd == 0:
        return pd.Series(np.nan, index=s.index)
    return (s - s.mean(skipna=True)) / sd


def winsorize(s, low=0.01, high=0.99):
    s = pd.to_numeric(s, errors="coerce")
    lo, hi = s.quantile(low), s.quantile(high)
    return s.clip(lo, hi)


def save_df(df, filename):
    path = RESULT_DIR / filename
    df.to_csv(path, index=False, encoding="utf-8-sig")
    print(f"[저장 완료] {path}")


def newey_west_mean_tstat(x, lag=6):
    """
    Fama-MacBeth 계수 평균의 Newey-West t-stat 계산.
    """
    x = pd.Series(x).dropna().astype(float)
    n = len(x)

    if n < 5:
        return np.nan, np.nan, np.nan

    xc = x - x.mean()
    gamma0 = np.sum(xc * xc) / n
    s = gamma0

    max_lag = min(lag, n - 1)

    for l in range(1, max_lag + 1):
        weight = 1 - l / (max_lag + 1)
        gamma_l = np.sum(xc.iloc[l:].values * xc.iloc[:-l].values) / n
        s += 2 * weight * gamma_l

    var_mean = s / n
    se = np.sqrt(var_mean) if var_mean > 0 else np.nan
    tstat = x.mean() / se if se and se > 0 else np.nan

    return x.mean(), se, tstat

# **3단계. 데이터 불러오기**

In [5]:
# ============================================================
# 3. 데이터 불러오기
# ============================================================

firm = read_csv_safely(FIRM_FILE)
news = read_csv_safely(NEWS_FILE)

firm = clean_columns(firm)
news = clean_columns(news)

firm = to_year_month(firm)
news = to_year_month(news)

print("기업데이터 shape:", firm.shape)
print("뉴스데이터 shape:", news.shape)

print("\n기업데이터 컬럼:")
print(firm.columns.tolist())

print("\n뉴스데이터 컬럼:")
print(news.columns.tolist())

기업데이터 shape: (253266, 60)
뉴스데이터 shape: (132, 33)

기업데이터 컬럼:
['KRX_Ticker', 'year', 'month', 'year_month', 'Company_Name_KRX', 'Market_Type_KRX', 'Price_Start', 'Price_End', 'Trading_Days', 'Trading_Volume', 'Trading_Value', 'MarketCap_EOM', 'ListedShares_EOM', 'Rank_EOM', 'monthly_return', 'Company_Name', 'Market_Type', 'DART_Corp_Code', 'Business_Registration_Number', 'KSIC_Code', 'Industry_DART_KSIC_Name', 'Industry_KRX', 'ETS_Entity_ID', 'ETS_Entity_Name', 'ETS_Dummy', 'ETS_Sector', 'CBAM_Dummy', 'CBAM_i', 'CBAM_Sector', 'EnergyIntensive_i', 'CarbonIntensive_i', 'EITE_Dummy', 'KAU_Price_Exposure', 'TransitionExposure_Simple', 'TransitionExposure_ZProxy', 'rf_annual_treasury3y_pct', 'rf_monthly_treasury3y', 'rf_annual_base_rate_pct', 'rf_monthly_base_rate', 'credit_spread_aa3y_minus_treasury3y_pct', 'inflation_proxy_cpi', 'excess_return_treasury3y', 'excess_return_base_rate', 'TotalAssets_lag1', 'TotalDebt_lag1', 'TotalEquity_lag1', 'Revenue_lag1', 'OperatingIncome_lag1', 'NetIncome_

# **4단계. 뉴스 기반 PolicyState 구축**

In [6]:
# ============================================================
# 4. 뉴스 기반 PolicyState 구축
# ============================================================

def build_policy_state(news):
    news = news.copy()

    # source 구분
    if "source" not in news.columns:
        news["source"] = "news"

    if "source_group" not in news.columns:
        source_group_col = find_col(news, ["source_gr", "source_group", "source"], required=False)
        if source_group_col:
            news["source_group"] = news[source_group_col]
        else:
            news["source_group"] = news["source"]

    # 사용 가능한 뉴스지수 컬럼 후보
    raw_score_map = {
        "trans": ["CRI_trans", "transition_score", "s_trans"],
        "phys": ["CRI_phys", "physical_score", "s_phys"],
        "cred": ["CRI_cred", "credibility_score", "s_cred"],
        "cost": ["CRI_cost", "carbon_cost_score", "s_cost"],
        "finance": ["CRI_finance", "finance_score", "s_fin"],
        "readiness": ["CRI_readiness", "readiness_score", "s_readiness"],
        "power": ["CRI_power", "power_allocation_score", "s_power"],
        "korea_policy": ["korea_policy_score", "CRI_korea_policy"],
    }

    agg_dict = {}

    for key, candidates in raw_score_map.items():
        col = find_col(news, candidates, required=False)
        if col:
            news[col] = pd.to_numeric(news[col], errors="coerce")
            agg_dict[f"CRI_{key}"] = (col, "mean")

    count_col = find_col(
        news,
        ["article_count", "included_article_count", "monthly_article_count"],
        required=False
    )

    if count_col:
        news[count_col] = pd.to_numeric(news[count_col], errors="coerce")
        agg_dict["article_count"] = (count_col, "sum")
    else:
        agg_dict["article_count"] = ("year_month", "size")

    # source-month 단위로 정리
    monthly_source = (
        news.groupby(["source_group", "year_month"])
        .agg(**agg_dict)
        .reset_index()
    )

    # source별 z-score
    cri_cols = [c for c in monthly_source.columns if c.startswith("CRI_")]

    for c in cri_cols:
        monthly_source[f"Z_{c}"] = monthly_source.groupby("source_group")[c].transform(zscore)

    # source 통합: 동일가중 평균
    z_cols = [f"Z_{c}" for c in cri_cols]

    monthly = (
        monthly_source.groupby("year_month")[z_cols + ["article_count"]]
        .agg({**{c: "mean" for c in z_cols}, "article_count": "sum"})
        .reset_index()
    )

    # 정책 이벤트 더미
    policy_event_months = [
        "2020-11", "2020-12", "2021-01",  # 2050 탄소중립 선언 전후
        "2021-01", "2021-02", "2021-03",  # ESG/기후공시 로드맵
        "2024-05", "2024-06", "2024-07",  # K-ETS 유동성·시장안정화 관련
        "2025-01", "2025-02", "2025-03",  # 2025 제도 시행 변화
    ]

    monthly["PolicyDummy"] = monthly["year_month"].isin(policy_event_months).astype(int)

    # 뉴스 기반 정책상태지수
    # 핵심: 정책 신뢰도, 탄소비용, 금융 관련 뉴스 강도
    policy_components = [
        "Z_CRI_cred",
        "Z_CRI_cost",
        "Z_CRI_finance",
        "Z_CRI_korea_policy",
    ]

    existing_policy_components = [c for c in policy_components if c in monthly.columns]

    if len(existing_policy_components) == 0:
        raise ValueError("PolicyState를 만들 수 있는 뉴스지수 컬럼이 없습니다.")

    monthly["PolicyState_news_proxy"] = monthly[existing_policy_components].mean(axis=1)
    monthly["PolicyState_news_proxy"] = zscore(monthly["PolicyState_news_proxy"])

    # event dummy 포함 버전
    monthly["Z_PolicyDummy"] = zscore(monthly["PolicyDummy"])
    monthly["PolicyState_event_augmented"] = (
        monthly[["PolicyState_news_proxy", "Z_PolicyDummy"]]
        .mean(axis=1)
    )
    monthly["PolicyState_event_augmented"] = zscore(monthly["PolicyState_event_augmented"])

    save_df(monthly_source, "news_source_monthly_indices.csv")
    save_df(monthly, "policy_state_monthly.csv")

    return monthly_source, monthly


monthly_source, policy = build_policy_state(news)

policy.head()

[저장 완료] /content/drive/MyDrive/박사논문/논문_climatefinance/data_climatefinance/results_colab/news_source_monthly_indices.csv
[저장 완료] /content/drive/MyDrive/박사논문/논문_climatefinance/data_climatefinance/results_colab/policy_state_monthly.csv


,year_month,Z_CRI_trans,Z_CRI_phys,Z_CRI_cred,Z_CRI_cost,Z_CRI_finance,Z_CRI_readiness,Z_CRI_power,article_count,PolicyDummy,PolicyState_news_proxy,Z_PolicyDummy,PolicyState_event_augmented
0,2015-01,-0.940984,-0.747942,-0.778890,-0.698971,-0.852351,-0.995690,-0.558053,1,0,-1.101317,-0.300367,-0.950812
1,2015-02,0.198607,-0.225392,2.497792,1.770717,-0.576522,-0.995690,0.521920,1,0,1.744926,-0.300367,0.979896
2,2015-03,0.383800,-0.747942,-0.881472,-0.270315,-0.852351,0.719194,-0.558053,1,0,-0.947205,-0.300367,-0.846273
3,2015-04,-0.771523,-0.225392,-0.783611,0.224072,-0.852351,0.674443,-0.558053,1,0,-0.667294,-0.300367,-0.656400
4,2015-05,0.380980,-0.747942,-1.196228,4.365122,-0.852351,-0.995690,-0.558053,1,0,1.094856,-0.300367,0.538930


# **5단계. 기업 변수 생성**

In [7]:
# ============================================================
# 5. 기업 변수 생성
# ============================================================

def build_firm_variables(firm):
    firm = firm.copy()

    # 기업 식별자
    entity_col = find_col(
        firm,
        ["KRX_Ticker", "ticker", "stock_code", "DART_Stock_Code", "Symbol"],
        required=True,
        label="기업 식별자"
    )

    # 월별 수익률
    ret_col = find_col(firm, ["monthly_return", "Monthly_Return"], required=False)

    if ret_col is None:
        price_start = find_col(firm, ["Price_Start", "price_start"], required=True, label="Price_Start")
        price_end = find_col(firm, ["Price_End", "price_end"], required=True, label="Price_End")
        firm["monthly_return"] = (
            pd.to_numeric(firm[price_end], errors="coerce")
            / pd.to_numeric(firm[price_start], errors="coerce")
            - 1
        )
    else:
        firm["monthly_return"] = pd.to_numeric(firm[ret_col], errors="coerce")

    # 무위험수익률
    rf_col = find_col(
        firm,
        ["rf_monthly_treasury3y", "rf_monthly", "rf_monthly_base_rate"],
        required=False
    )

    if rf_col:
        firm["rf_monthly"] = pd.to_numeric(firm[rf_col], errors="coerce")
    else:
        firm["rf_monthly"] = 0.0
        print("[주의] 무위험수익률 컬럼이 없어 0으로 처리했습니다.")

    # 초과수익률
    excess_col = find_col(
        firm,
        ["excess_return_treasury3y", "excess_return", "excess_ret"],
        required=False
    )

    if excess_col:
        firm["excess_return"] = pd.to_numeric(firm[excess_col], errors="coerce")
    else:
        firm["excess_return"] = firm["monthly_return"] - firm["rf_monthly"]

    # 전환노출도 구성
    if "TransitionExposure_Simple" in firm.columns:
        firm["TransitionExposure_raw"] = pd.to_numeric(firm["TransitionExposure_Simple"], errors="coerce")
    elif "Transition" in firm.columns:
        firm["TransitionExposure_raw"] = pd.to_numeric(firm["Transition"], errors="coerce")
    else:
        exposure_candidates = [
            "ETS_Dummy",
            "CBAM_Dummy",
            "EnergyIntensive_Dummy",
            "EnergyIntensive",
            "CarbonIntensive_Dummy",
            "CarbonIntensive",
            "EITE_Dummy",
            "KAU_Exposure_Dummy",
        ]

        exposure_cols = []
        for c in exposure_candidates:
            found = find_col(firm, [c], required=False)
            if found:
                exposure_cols.append(found)

        if len(exposure_cols) == 0:
            raise ValueError("전환노출도 구성용 컬럼이 없습니다. ETS_Dummy, CBAM_Dummy 등을 확인하세요.")

        for c in exposure_cols:
            firm[c] = pd.to_numeric(firm[c], errors="coerce")

        firm["TransitionExposure_raw"] = firm[exposure_cols].mean(axis=1)

    firm["TransitionExposure"] = zscore(firm["TransitionExposure_raw"])

    # TRI 전환준비도
    tri_existing = find_col(
        firm,
        ["TRI", "Transition_Readiness", "TransitionReadiness", "Transition_Readiness_lag1"],
        required=False
    )

    if tri_existing:
        firm["TRI_raw"] = pd.to_numeric(firm[tri_existing], errors="coerce")
    else:
        tri_components = [
            "GoalSpecificity",
            "ImplementationPlan",
            "CarbonRiskDisclosure",
            "GreenCapexCommitment",
            "RenewableProcurement",
        ]

        tri_cols = []
        for c in tri_components:
            found = find_col(firm, [c], required=False)
            if found:
                tri_cols.append(found)

        if len(tri_cols) > 0:
            for c in tri_cols:
                firm[c] = pd.to_numeric(firm[c], errors="coerce")
            firm["TRI_raw"] = firm[tri_cols].mean(axis=1)
        else:
            firm["TRI_raw"] = np.nan
            print("[주의] TRI 관련 컬럼이 없습니다. 전환준비도 분석은 TRI 입력 후 실행해야 합니다.")

    firm["TRI"] = zscore(firm["TRI_raw"])

    # 주요 통제변수
    control_candidates = {
        "Size": ["Size", "Size_lag1", "MarketCap", "Market_Cap"],
        "Leverage": ["Leverage", "Leverage_lag1"],
        "Profitability": ["Profitability", "Profitability_lag1"],
        "DebtCost": ["DebtCost", "DebtCost_lag1"],
    }

    for new_col, candidates in control_candidates.items():
        found = find_col(firm, candidates, required=False)
        if found:
            firm[new_col] = pd.to_numeric(firm[found], errors="coerce")
        else:
            firm[new_col] = np.nan

    # 날짜 정렬
    firm["date_m"] = pd.to_datetime(firm["year_month"] + "-01", errors="coerce")
    firm = firm.sort_values([entity_col, "date_m"]).copy()

    # lag 처리
    for c in ["TransitionExposure", "TRI", "Size", "Leverage", "Profitability", "DebtCost"]:
        firm[f"{c}_lag1"] = firm.groupby(entity_col)[c].shift(1)

    # 미래 초과수익률
    firm["excess_return_fwd1"] = firm.groupby(entity_col)["excess_return"].shift(-1)

    # Momentum 12-2
    def calc_momentum(x):
        return (1 + x.shift(2)).rolling(11).apply(np.prod, raw=True) - 1

    firm["Momentum_12_2"] = firm.groupby(entity_col)["monthly_return"].transform(calc_momentum)

    # winsorization
    for c in [
        "monthly_return", "excess_return", "excess_return_fwd1",
        "Size_lag1", "Leverage_lag1", "Profitability_lag1",
        "DebtCost_lag1", "Momentum_12_2"
    ]:
        if c in firm.columns:
            firm[c] = winsorize(firm[c])

    return firm, entity_col


firm2, entity_col = build_firm_variables(firm)

print("기업 식별자:", entity_col)
firm2.head()

기업 식별자: KRX_Ticker


,KRX_Ticker,year,month,year_month,Company_Name_KRX,Market_Type_KRX,Price_Start,Price_End,Trading_Days,Trading_Volume,...,TRI,Size,Leverage,Profitability,DebtCost,date_m,TransitionExposure_lag1,TRI_lag1,excess_return_fwd1,Momentum_12_2
0,20,2015,1,201501,동화약품,KOSPI,5530.0,5980.0,21,1660862.0,...,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaN,0.055879,NaN
1,20,2015,2,201502,동화약품,KOSPI,5910.0,6250.0,17,1992086.0,...,NaN,NaN,NaN,NaN,NaN,NaT,-0.348341,NaN,0.201994,NaN
2,20,2015,3,201503,동화약품,KOSPI,6240.0,7510.0,22,3713992.0,...,NaN,NaN,NaN,NaN,NaN,NaT,-0.348341,NaN,0.084630,NaN
3,20,2015,4,201504,동화약품,KOSPI,7670.0,8330.0,22,4817514.0,...,NaN,NaN,NaN,NaN,NaN,NaT,-0.348341,NaN,0.163968,NaN
4,20,2015,5,201505,동화약품,KOSPI,8040.0,9370.0,18,5986420.0,...,NaN,NaN,NaN,NaN,NaN,NaT,-0.348341,NaN,-0.064375,NaN


# **6단계. 뉴스 정책상태지수와 기업패널 병합**

In [8]:
# ============================================================
# 6. 기업-월 패널과 PolicyState 병합
# ============================================================

panel = firm2.merge(
    policy[[
        "year_month",
        "PolicyState_news_proxy",
        "PolicyState_event_augmented",
        "PolicyDummy"
    ]],
    on="year_month",
    how="left"
)

# 기본 PolicyState 선택
panel["PolicyState"] = panel["PolicyState_event_augmented"]
panel["PolicyState"] = zscore(panel["PolicyState"])

# 중심화
for c in ["TransitionExposure_lag1", "TRI_lag1", "PolicyState"]:
    panel[f"{c}_c"] = panel[c] - panel[c].mean(skipna=True)

# 상호작용항
panel["TE_x_PS"] = panel["TransitionExposure_lag1_c"] * panel["PolicyState_c"]
panel["TE_x_TRI"] = panel["TransitionExposure_lag1_c"] * panel["TRI_lag1_c"]
panel["TRI_x_PS"] = panel["TRI_lag1_c"] * panel["PolicyState_c"]
panel["TE_x_TRI_x_PS"] = (
    panel["TransitionExposure_lag1_c"]
    * panel["TRI_lag1_c"]
    * panel["PolicyState_c"]
)

# ETS/CBAM validation용 상호작용
ets_col = find_col(panel, ["ETS_Dummy"], required=False)
cbam_col = find_col(panel, ["CBAM_Dummy"], required=False)

if ets_col:
    panel["ETS_x_PS"] = pd.to_numeric(panel[ets_col], errors="coerce") * panel["PolicyState_c"]

if cbam_col:
    panel["CBAM_x_PS"] = pd.to_numeric(panel[cbam_col], errors="coerce") * panel["PolicyState_c"]

save_df(panel, "analysis_panel_merged.csv")

print("최종 패널 shape:", panel.shape)
panel.head()

[저장 완료] /content/drive/MyDrive/박사논문/논문_climatefinance/data_climatefinance/results_colab/analysis_panel_merged.csv
최종 패널 shape: (253266, 88)


,KRX_Ticker,year,month,year_month,Company_Name_KRX,Market_Type_KRX,Price_Start,Price_End,Trading_Days,Trading_Volume,...,PolicyState,TransitionExposure_lag1_c,TRI_lag1_c,PolicyState_c,TE_x_PS,TE_x_TRI,TRI_x_PS,TE_x_TRI_x_PS,ETS_x_PS,CBAM_x_PS
0,20,2015,1,201501,동화약품,KOSPI,5530.0,5980.0,21,1660862.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,20,2015,2,201502,동화약품,KOSPI,5910.0,6250.0,17,1992086.0,...,NaN,-0.348758,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,20,2015,3,201503,동화약품,KOSPI,6240.0,7510.0,22,3713992.0,...,NaN,-0.348758,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,20,2015,4,201504,동화약품,KOSPI,7670.0,8330.0,22,4817514.0,...,NaN,-0.348758,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,20,2015,5,201505,동화약품,KOSPI,8040.0,9370.0,18,5986420.0,...,NaN,-0.348758,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [11]:
print(news.columns.tolist())
print(news[["year", "month"]].head() if {"year", "month"}.issubset(news.columns) else news.head())

possible_cols = [
    "CRI_cred", "CRI_cost", "CRI_finance", "CRI_readiness",
    "Z_CRI_cred", "Z_CRI_cost", "Z_CRI_finance", "Z_CRI_readiness",
    "PolicyState", "PolicyState_news_proxy", "PolicyState_formula_ready"
]

for c in possible_cols:
    if c in news.columns:
        print(c, news[c].notna().sum(), news[c].head().tolist())

['month', 'year', 'source_count', 'source_weighting', 'CRI_trans_integrated', 'CRI_phys_integrated', 'CRI_cred_integrated', 'CRI_cost_integrated', 'CRI_finance_integrated', 'CRI_readiness_integrated', 'CRI_power_integrated', 'Z_CRI_cred_integrated', 'Z_CRI_cost_integrated', 'Z_CRI_finance_integrated', 'PolicyState_news_proxy', 'PolicyDummy', 'KAUPrice', 'Z_KAUPrice', 'PolicyState_formula_ready', 'naver_Z_CRI_cred', 'naver_Z_CRI_cost', 'naver_Z_CRI_finance', 'PolicyState_news_proxy_naver', 'guardian_Z_CRI_cred', 'guardian_Z_CRI_cost', 'guardian_Z_CRI_finance', 'PolicyState_news_proxy_guardian', 'gdelt_Z_CRI_cred', 'gdelt_Z_CRI_cost', 'gdelt_Z_CRI_finance', 'PolicyState_news_proxy_gdelt', 'dataset_generated_at', 'year_month']
   year    month
0  2015  2015-01
1  2015  2015-02
2  2015  2015-03
3  2015  2015-04
4  2015  2015-05
PolicyState_news_proxy 132 [-2.339089536, 3.706052705, -2.011772575, -1.417268709, 2.325367798]
PolicyState_formula_ready 0 [nan, nan, nan, nan, nan]


# **7단계. 기초통계와 상관관계**

In [9]:
# ============================================================
# 7. 기초통계와 상관관계
# ============================================================

desc_vars = [
    "excess_return",
    "excess_return_fwd1",
    "TransitionExposure_lag1",
    "TRI_lag1",
    "PolicyState",
    "TE_x_PS",
    "TE_x_TRI_x_PS",
    "Size_lag1",
    "Leverage_lag1",
    "Profitability_lag1",
    "DebtCost_lag1",
    "Momentum_12_2",
]

desc_vars = [c for c in desc_vars if c in panel.columns]

desc = panel[desc_vars].describe(percentiles=[0.01, 0.25, 0.5, 0.75, 0.99]).T
desc["missing"] = panel[desc_vars].isna().sum()

save_df(desc.reset_index().rename(columns={"index": "variable"}), "table1_descriptive_statistics.csv")

corr = panel[desc_vars].corr()
save_df(corr.reset_index().rename(columns={"index": "variable"}), "table2_correlation_matrix.csv")

desc

[저장 완료] /content/drive/MyDrive/박사논문/논문_climatefinance/data_climatefinance/results_colab/table1_descriptive_statistics.csv
[저장 완료] /content/drive/MyDrive/박사논문/논문_climatefinance/data_climatefinance/results_colab/table2_correlation_matrix.csv


,count,mean,std,min,1%,25%,50%,75%,99%,max,missing
excess_return,253266.0,0.001349,0.139130,-0.319377,-0.319377,-0.074527,-0.009968,0.055016,0.580269,0.580270,0
excess_return_fwd1,250887.0,0.001066,0.138461,-0.317498,-0.317492,-0.074606,-0.010190,0.054570,0.575988,0.576012,2379
TransitionExposure_lag1,250887.0,0.000418,1.000523,-0.348341,-0.348341,-0.348341,-0.348341,-0.348341,4.523051,5.497329,2379
TRI_lag1,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,253266
PolicyState,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,253266
TE_x_PS,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,253266
TE_x_TRI_x_PS,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,253266
Size_lag1,225730.0,26.122638,1.452300,23.359004,23.359004,25.126176,25.876868,26.874213,31.076549,31.076549,27536
Leverage_lag1,225707.0,0.523725,0.298541,0.053751,0.053751,0.278414,0.478153,0.722952,1.000000,1.000000,27559
Profitability_lag1,224097.0,-0.000446,0.114556,-0.535144,-0.535144,-0.020111,0.020613,0.057011,0.220431,0.220431,29169


8단계. 패널 고정효과 회귀

핵심 회귀식입니다.

R
i,t+1


−R
f,t+1


=α
i


+τ
t


+β
1


TransitionExposure
i,t


+β
2


TRI
i,t


+β
3


TransitionExposure
i,t


×PolicyState
t


+β
4


# TransitionExpos **굵은 텍스트**

In [10]:
# ============================================================
# 8. 패널 고정효과 회귀
# ============================================================

def run_panel_fe(data, y, xvars, entity_col, result_name):
    use_cols = [entity_col, "year_month", y] + xvars
    tmp = data[use_cols].dropna().copy()

    if tmp.empty:
        print(f"[스킵] {result_name}: 유효 표본이 없습니다.")
        return None

    tmp["time"] = pd.to_datetime(tmp["year_month"] + "-01", errors="coerce")
    tmp = tmp.set_index([entity_col, "time"]).sort_index()

    y_data = tmp[y]
    X = tmp[xvars]

    model = PanelOLS(
        y_data,
        X,
        entity_effects=True,
        time_effects=True,
        drop_absorbed=True,
        check_rank=False
    )

    res = model.fit(
        cov_type="clustered",
        cluster_entity=True,
        cluster_time=True
    )

    coef = pd.DataFrame({
        "coef": res.params,
        "std_err": res.std_errors,
        "tstat": res.tstats,
        "pvalue": res.pvalues
    })

    save_df(coef.reset_index().rename(columns={"index": "variable"}), f"{result_name}.csv")

    with open(RESULT_DIR / f"{result_name}.txt", "w", encoding="utf-8") as f:
        f.write(str(res.summary))

    print(res.summary)
    return res


controls = [
    "Size_lag1",
    "Leverage_lag1",
    "Profitability_lag1",
    "DebtCost_lag1",
    "Momentum_12_2",
]
controls = [c for c in controls if c in panel.columns]

# Model 1: base
x_m1 = [
    "TransitionExposure_lag1_c",
    "TRI_lag1_c",
] + controls

# Model 2: policy interaction
x_m2 = [
    "TransitionExposure_lag1_c",
    "TRI_lag1_c",
    "TE_x_PS",
] + controls

# Model 3: full model
x_m3 = [
    "TransitionExposure_lag1_c",
    "TRI_lag1_c",
    "TE_x_PS",
    "TE_x_TRI_x_PS",
] + controls

# 현재월 realized return
res_current = run_panel_fe(
    panel,
    y="excess_return",
    xvars=x_m3,
    entity_col=entity_col,
    result_name="panel_current_full"
)

# 다음월 수익률: 기대수익률 해석에 더 적합
res_fwd1_base = run_panel_fe(
    panel,
    y="excess_return_fwd1",
    xvars=x_m1,
    entity_col=entity_col,
    result_name="panel_fwd1_base"
)

res_fwd1_policy = run_panel_fe(
    panel,
    y="excess_return_fwd1",
    xvars=x_m2,
    entity_col=entity_col,
    result_name="panel_fwd1_policy_interaction"
)

res_fwd1_full = run_panel_fe(
    panel,
    y="excess_return_fwd1",
    xvars=x_m3,
    entity_col=entity_col,
    result_name="panel_fwd1_full"
)

[스킵] panel_current_full: 유효 표본이 없습니다.
[스킵] panel_fwd1_base: 유효 표본이 없습니다.
[스킵] panel_fwd1_policy_interaction: 유효 표본이 없습니다.
[스킵] panel_fwd1_full: 유효 표본이 없습니다.
